# INGD Training Notebook for Kaggle

This notebook trains Neural Granger models for the Fault.ai application.

## Setup
1. Enable GPU: Settings > Accelerator > GPU T4 x2
2. Add dataset (Train-Ticket or GAIA) to the notebook
3. Run all cells

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Imports
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# Paths
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")
WEIGHTS_DIR = KAGGLE_OUTPUT / "weights"
LOGS_DIR = KAGGLE_OUTPUT / "logs"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# List available datasets
print("Available datasets:")
for d in KAGGLE_INPUT.iterdir():
    print(f"  - {d.name}")

In [ ]:
@dataclass
class TrainingConfig:
    """Configuration for INGD training."""
    hidden_dim: int = 64
    num_layers: int = 2
    dropout: float = 0.1
    max_lag: int = 5
    learning_rate: float = 0.001
    lambda_sparse: float = 0.01
    batch_size: int = 32
    num_epochs: int = 100
    early_stopping_patience: int = 10
    top_k: int = 5

config = TrainingConfig()
print("Training Configuration:")
for k, v in asdict(config).items():
    print(f"  {k}: {v}")

In [ ]:
class MLPGranger(torch.nn.Module):
    """MLP-based Neural Granger model."""

    def __init__(self, num_series, max_lag=5, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_series = num_series
        self.max_lag = max_lag
        self.hidden_dim = hidden_dim

        input_dim = num_series * max_lag
        self.input_layer = torch.nn.Linear(input_dim, hidden_dim)

        layers = []
        for _ in range(num_layers - 1):
            layers.extend([
                torch.nn.Linear(hidden_dim, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout)
            ])
        self.hidden_layers = torch.nn.Sequential(*layers) if layers else torch.nn.Identity()
        self.output_layer = torch.nn.Linear(hidden_dim, num_series)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    torch.nn.init.zeros_(m.bias)

    def forward(self, x):
        h = torch.nn.functional.relu(self.input_layer(x))
        h = self.hidden_layers(h)
        return self.output_layer(h)

    def get_causal_weights(self):
        weights = self.input_layer.weight.data
        weights = weights.view(self.hidden_dim, self.num_series, self.max_lag)
        output_weights = self.output_layer.weight.data

        causal_matrix = torch.zeros(self.num_series, self.num_series)
        for i in range(self.num_series):
            for j in range(self.num_series):
                w = torch.abs(output_weights[i]).unsqueeze(1) * torch.abs(weights[:, j, :])
                causal_matrix[i, j] = w.sum()

        if causal_matrix.max() > 0:
            causal_matrix = causal_matrix / causal_matrix.max()
        return causal_matrix

    def group_lasso_penalty(self):
        weights = self.input_layer.weight.view(self.hidden_dim, self.num_series, self.max_lag)
        group_norms = torch.sqrt((weights ** 2).sum(dim=(0, 2)) + 1e-8)
        return group_norms.sum()

In [ ]:
def create_lagged_features(data, max_lag):
    """Create lagged feature matrix."""
    X_list = [data[max_lag - lag:-lag] for lag in range(1, max_lag + 1)]
    X = np.concatenate(X_list, axis=1)
    y = data[max_lag:]
    return X, y

def preprocess_data(data):
    """Preprocess metrics data."""
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0) + 1e-8
    data = (data - mean) / std
    return np.clip(data, -5, 5)

In [ ]:
def train_model(data, metric_names, config, device=DEVICE, verbose=True):
    """Train Neural Granger model."""
    num_metrics = data.shape[1]
    data = preprocess_data(data)
    X, y = create_lagged_features(data, config.max_lag)

    X_tensor = torch.FloatTensor(X).to(device)
    y_tensor = torch.FloatTensor(y).to(device)

    model = MLPGranger(
        num_series=num_metrics,
        max_lag=config.max_lag,
        hidden_dim=config.hidden_dim,
        num_layers=config.num_layers,
        dropout=config.dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    criterion = torch.nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

    history = {"loss": [], "recon_loss": [], "sparse_loss": []}
    best_loss = float('inf')
    patience_counter = 0

    epoch_iter = tqdm(range(config.num_epochs), desc="Training") if verbose else range(config.num_epochs)

    for epoch in epoch_iter:
        model.train()
        epoch_loss, epoch_recon, epoch_sparse = 0, 0, 0

        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            recon_loss = criterion(predictions, batch_y)
            sparse_loss = config.lambda_sparse * model.group_lasso_penalty()
            loss = recon_loss + sparse_loss
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_sparse += sparse_loss.item()

        n = len(dataloader)
        history["loss"].append(epoch_loss / n)
        history["recon_loss"].append(epoch_recon / n)
        history["sparse_loss"].append(epoch_sparse / n)

        if history["loss"][-1] < best_loss:
            best_loss = history["loss"][-1]
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.early_stopping_patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}")
                break

        if verbose:
            epoch_iter.set_postfix({"loss": f"{history['loss'][-1]:.4f}"})

    model.eval()
    with torch.no_grad():
        causal_matrix = model.get_causal_weights().cpu().numpy()

    return model, causal_matrix, history

In [ ]:
def evaluate_root_cause(causal_matrix, metric_names, ground_truth, top_k=5):
    """Evaluate root cause detection."""
    out_degree = causal_matrix.sum(axis=1)
    in_degree = causal_matrix.sum(axis=0)
    scores = out_degree / (in_degree + 1)

    ranked_indices = np.argsort(-scores)
    ranked_names = [metric_names[i] for i in ranked_indices]

    gt_lower = ground_truth.lower()
    rank = -1
    for i, name in enumerate(ranked_names):
        if gt_lower in name.lower() or name.lower() in gt_lower:
            rank = i + 1
            break

    return {
        "ground_truth": ground_truth,
        "top_k": ranked_names[:top_k],
        "hit_at_1": rank == 1,
        "hit_at_3": 1 <= rank <= 3,
        "hit_at_5": 1 <= rank <= 5,
        "rank": rank
    }

## Generate Synthetic Data for Testing

In [ ]:
# Generate synthetic data
np.random.seed(42)

num_services = 10
num_timesteps = 200
fault_service = 2
fault_start = 100

service_names = [f"ts-service-{i}" for i in range(num_services)]
service_names[fault_service] = "ts-order-service"

# Generate metrics
base_latency = np.random.uniform(10, 50, num_services)
metrics = np.zeros((num_timesteps, num_services))

for t in range(num_timesteps):
    metrics[t] = base_latency + np.random.randn(num_services) * 5

# Inject fault
for t in range(fault_start, num_timesteps):
    metrics[t, fault_service] *= (1 + 2 * np.random.rand())
    for dep in [(fault_service + i) % num_services for i in range(1, 4)]:
        if t >= fault_start + 5:
            metrics[t, dep] *= (1 + np.random.rand())

print(f"Generated synthetic data: {metrics.shape}")
print(f"Services: {service_names}")
print(f"Fault injected in: {service_names[fault_service]}")

In [ ]:
# Visualize metrics
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(metrics[:, i])
    ax.axvline(x=fault_start, color='r', linestyle='--', alpha=0.5)
    ax.set_title(service_names[i])
    ax.set_xlabel("Time")
plt.suptitle("Service Metrics (red line = fault injection)")
plt.tight_layout()
plt.show()

## Train Model

In [ ]:
# Train
model, causal_matrix, history = train_model(metrics, service_names, config)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["loss"])
axes[0].set_title("Total Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["recon_loss"])
axes[1].set_title("Reconstruction Loss")
axes[1].set_xlabel("Epoch")

axes[2].plot(history["sparse_loss"])
axes[2].set_title("Sparsity Loss")
axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize causal matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    causal_matrix,
    xticklabels=service_names,
    yticklabels=service_names,
    cmap="Reds",
    annot=True,
    fmt=".2f"
)
plt.title("Learned Causal Matrix")
plt.xlabel("Effect (Target)")
plt.ylabel("Cause (Source)")
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate
eval_result = evaluate_root_cause(causal_matrix, service_names, "ts-order-service")

print("\n" + "="*50)
print("Evaluation Results")
print("="*50)
print(f"Ground Truth: {eval_result['ground_truth']}")
print(f"Top 5 Predictions: {eval_result['top_k']}")
print(f"Hit@1: {eval_result['hit_at_1']}")
print(f"Hit@3: {eval_result['hit_at_3']}")
print(f"Hit@5: {eval_result['hit_at_5']}")
print(f"Rank: {eval_result['rank']}")

## Save Model

In [ ]:
# Save model
model_path = WEIGHTS_DIR / "neural_granger_synthetic.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "num_series": model.num_series,
    "max_lag": model.max_lag,
    "hidden_dim": model.hidden_dim,
    "config": asdict(config),
    "history": history,
    "metric_names": service_names
}, model_path)

print(f"Model saved to {model_path}")

# Save config
config_path = WEIGHTS_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(asdict(config), f, indent=2)

print(f"Config saved to {config_path}")

In [ ]:
# List output files
print("\nOutput files:")
for f in WEIGHTS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## Download Models

After running this notebook, download the files from `/kaggle/working/weights/` and place them in your Fault.ai app's `backend/weights/` folder.